# Fine-Tuning from Scratch

**Module 09: Fine-Tuning**  
**Objective**: Master parameter-efficient fine-tuning techniques

Fine-tuning approaches:
- Full fine-tuning
- LoRA (Low-Rank Adaptation)
- QLoRA (Quantized LoRA)
- Prefix tuning
- Adapter layers

## What You'll Learn
1. Full fine-tuning vs parameter-efficient methods
2. LoRA mathematics and implementation
3. QLoRA for large models
4. Prefix tuning and prompt tuning
5. Adapter layers
6. Trade-offs: performance vs efficiency

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)

## 1. Full Fine-Tuning vs Parameter-Efficient

**Problem**: Fine-tuning all parameters of large models is expensive
- GPT-3 (175B params): ~700GB in FP32
- Requires storing optimizer states (2-3× model size)
- Memory: 700GB model + 1.4TB optimizer = 2.1TB total!

**Solution**: Parameter-Efficient Fine-Tuning (PEFT)
- Update only small subset of parameters
- Rest frozen → huge memory savings

In [ ]:
def compare_fine_tuning_approaches():
    """Compare memory and trainable parameters."""
    
    # Model sizes (billions of parameters)
    model_sizes = [1, 7, 13, 30, 70, 175]
    
    # Memory requirements (GB)
    # Full fine-tuning: model + gradients + optimizer (Adam: 2 states)
    full_memory = [size * 4 * 4 for size in model_sizes]  # 4 copies in FP32
    
    # LoRA: frozen model + small trainable params + optimizer for trainable
    # Assume 0.1% trainable params
    lora_memory = [size * 4 * 1.01 for size in model_sizes]  # ~1% overhead
    
    # Trainable parameters (millions)
    full_trainable = [size * 1000 for size in model_sizes]  # All params
    lora_trainable = [size * 1000 * 0.001 for size in model_sizes]  # 0.1%
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Memory comparison
    x = np.arange(len(model_sizes))
    width = 0.35
    
    axes[0].bar(x - width/2, full_memory, width, label='Full Fine-tuning', color='lightcoral')
    axes[0].bar(x + width/2, lora_memory, width, label='LoRA', color='lightgreen')
    
    axes[0].set_xlabel('Model Size (B params)')
    axes[0].set_ylabel('Peak Memory (GB)')
    axes[0].set_title('Memory Requirements')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'{s}B' for s in model_sizes])
    axes[0].legend()
    axes[0].set_yscale('log')
    axes[0].grid(axis='y', alpha=0.3)
    
    # Trainable parameters
    axes[1].bar(x - width/2, full_trainable, width, label='Full Fine-tuning', color='lightcoral')
    axes[1].bar(x + width/2, lora_trainable, width, label='LoRA', color='lightgreen')
    
    axes[1].set_xlabel('Model Size (B params)')
    axes[1].set_ylabel('Trainable Parameters (M)')
    axes[1].set_title('Parameters to Update')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f'{s}B' for s in model_sizes])
    axes[1].legend()
    axes[1].set_yscale('log')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nFull Fine-tuning:")
    print("  ✓ Pros: Maximum flexibility, best performance potential")
    print("  ✗ Cons: Huge memory (4× model size), slow, expensive")
    print("\nLoRA:")
    print("  ✓ Pros: 100-1000× fewer trainable params, ~same memory as inference")
    print("  ✓ Pros: Fast training, multiple adapters for one base model")
    print("  ✗ Cons: Slightly lower performance ceiling")
    
    # Specific examples
    print("\n" + "="*70)
    print("Example: Fine-tuning LLaMA-7B")
    print(f"  Full fine-tuning: {7 * 4 * 4:.0f}GB memory, 7,000M trainable params")
    print(f"  LoRA: {7 * 4 * 1.01:.0f}GB memory, 7M trainable params (1000× reduction!)")
    print("="*70)

compare_fine_tuning_approaches()

## 2. LoRA (Low-Rank Adaptation)

**Key Idea**: Weight updates have low "intrinsic rank"

Instead of updating $W \in \mathbb{R}^{d \times k}$:

$$W_{new} = W + \Delta W$$

Decompose update as low-rank:

$$\Delta W = BA$$

where:
- $B \in \mathbb{R}^{d \times r}$  
- $A \in \mathbb{R}^{r \times k}$  
- $r \ll \min(d, k)$ (rank, typically 4-64)

**Parameters**:
- Original: $d \times k$
- LoRA: $r \times (d + k)$

**For $d=4096, k=4096, r=8$**:
- Original: 16M params
- LoRA: 65K params (250× reduction!)

In [ ]:
class LoRALayer(nn.Module):
    """
    LoRA layer for any linear transformation.
    
    Forward: y = Wx + (BA)x = Wx + B(Ax)
    where W is frozen, B and A are trainable.
    """
    
    def __init__(self, in_features, out_features, rank=8, alpha=16, dropout=0.1):
        super().__init__()
        
        self.rank = rank
        self.alpha = alpha  # Scaling factor
        self.scaling = alpha / rank
        
        # Frozen pretrained weight
        self.weight = nn.Parameter(torch.randn(out_features, in_features), requires_grad=False)
        
        # LoRA matrices (trainable)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))  # Init B to zero
        
        self.dropout = nn.Dropout(dropout)
        
        # Initialize A with Kaiming uniform
        nn.init.kaiming_uniform_(self.lora_A, a=np.sqrt(5))
    
    def forward(self, x):
        # Original transformation (frozen)
        result = F.linear(x, self.weight)
        
        # LoRA transformation (trainable)
        # x: (batch, seq, in_features)
        # A: (rank, in_features)
        # B: (out_features, rank)
        lora_out = self.dropout(x @ self.lora_A.T)  # (batch, seq, rank)
        lora_out = lora_out @ self.lora_B.T  # (batch, seq, out_features)
        lora_out = lora_out * self.scaling
        
        return result + lora_out
    
    def merge_weights(self):
        """Merge LoRA weights into base weight for inference."""
        self.weight.data += (self.lora_B @ self.lora_A) * self.scaling
        # Can now delete lora_A and lora_B

# Example usage
d_model = 4096
lora_layer = LoRALayer(d_model, d_model, rank=8)

# Count parameters
original_params = d_model * d_model
lora_params = lora_layer.lora_A.numel() + lora_layer.lora_B.numel()
reduction = original_params / lora_params

print(f"Original parameters: {original_params:,}")
print(f"LoRA parameters: {lora_params:,}")
print(f"Reduction: {reduction:.1f}×")
print(f"Trainable fraction: {lora_params/original_params*100:.3f}%")

# Test forward pass
x = torch.randn(2, 512, d_model)
output = lora_layer(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

## 3. LoRA in Attention

Apply LoRA to query and value projections (most impactful):

$$Q = W_Q x + B_Q A_Q x$$
$$V = W_V x + B_V A_V x$$

In [ ]:
class MultiHeadAttentionWithLoRA(nn.Module):
    """Multi-head attention with LoRA on Q and V projections."""
    
    def __init__(self, d_model, n_heads, lora_rank=8, lora_alpha=16):
        super().__init__()
        
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_model = d_model
        
        # Q, K, V projections with LoRA
        self.q_proj = LoRALayer(d_model, d_model, rank=lora_rank, alpha=lora_alpha)
        self.k_proj = nn.Linear(d_model, d_model)  # K often not adapted
        self.v_proj = LoRALayer(d_model, d_model, rank=lora_rank, alpha=lora_alpha)
        
        self.out_proj = nn.Linear(d_model, d_model)
        
        # Freeze K and out_proj for demonstration
        for param in self.k_proj.parameters():
            param.requires_grad = False
        for param in self.out_proj.parameters():
            param.requires_grad = False
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()
        
        # Project Q, K, V
        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)
        
        # Reshape for multi-head attention
        Q = Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = F.softmax(scores, dim=-1)
        
        # Apply attention to values
        out = torch.matmul(attn, V)
        
        # Concatenate heads
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # Output projection
        out = self.out_proj(out)
        
        return out

# Create attention layer with LoRA
d_model = 768
n_heads = 12
attn_lora = MultiHeadAttentionWithLoRA(d_model, n_heads, lora_rank=8).to(device)

# Count trainable parameters
total_params = sum(p.numel() for p in attn_lora.parameters())
trainable_params = sum(p.numel() for p in attn_lora.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable: {trainable_params/total_params*100:.2f}%")

# Test
x = torch.randn(2, 128, d_model).to(device)
output = attn_lora(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

## 4. Visualizing LoRA Rank

**Question**: How does rank $r$ affect performance and parameters?

In [ ]:
def visualize_lora_rank_tradeoff():
    """Visualize rank vs parameters and performance."""
    
    d_model = 4096
    ranks = [1, 2, 4, 8, 16, 32, 64, 128]
    
    # Parameters for single layer
    original_params = d_model * d_model
    lora_params = [r * (d_model + d_model) for r in ranks]
    param_ratios = [lp / original_params * 100 for lp in lora_params]
    
    # Simulated performance (accuracy)
    # Lower rank = less capacity, higher rank = more capacity (diminishing returns)
    performance = [75, 80, 85, 88, 90, 91, 91.5, 91.7]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Parameters vs rank
    axes[0].plot(ranks, param_ratios, marker='o', linewidth=2, color='blue')
    axes[0].set_xlabel('LoRA Rank (r)', fontsize=12)
    axes[0].set_ylabel('Trainable Parameters (%)', fontsize=12)
    axes[0].set_title('Parameters vs Rank')
    axes[0].set_xscale('log', base=2)
    axes[0].grid(True, alpha=0.3)
    
    # Add annotations
    for i, (r, ratio) in enumerate(zip(ranks, param_ratios)):
        if i % 2 == 0:
            axes[0].annotate(f'r={r}\n{ratio:.2f}%', 
                           (r, ratio), xytext=(10, 10), 
                           textcoords='offset points', fontsize=9)
    
    # Performance vs rank
    axes[1].plot(ranks, performance, marker='s', linewidth=2, color='green')
    axes[1].axhline(y=91, color='red', linestyle='--', label='Diminishing returns', alpha=0.5)
    axes[1].set_xlabel('LoRA Rank (r)', fontsize=12)
    axes[1].set_ylabel('Task Accuracy (%)', fontsize=12)
    axes[1].set_title('Performance vs Rank')
    axes[1].set_xscale('log', base=2)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nLoRA Rank Selection:")
    print("  • r=4-8: Good balance for most tasks (0.2-0.4% params)")
    print("  • r=16-32: High-capacity tasks (0.8-1.5% params)")
    print("  • r>64: Diminishing returns")
    print("\n✓ Sweet spot: r=8 (0.4% params, ~90% of full fine-tuning performance)")

visualize_lora_rank_tradeoff()

## 5. QLoRA (Quantized LoRA)

**Problem**: Even with LoRA, base model still uses lots of memory

**Solution**: Quantize base model to 4-bit, train LoRA in higher precision

**QLoRA Pipeline**:
1. Load base model in 4-bit (NF4 format)
2. Add LoRA adapters (FP16/BF16)
3. Train only LoRA parameters
4. Gradients computed in FP16, base model stays 4-bit

**Memory savings**: ~4× additional reduction

In [ ]:
class NF4Quantization:
    """
    NormalFloat4 (NF4) quantization used in QLoRA.
    
    Key idea: Quantize to 4 bits using information-theoretically optimal bins
    for normally distributed weights.
    """
    
    def __init__(self):
        # NF4 quantization bins (16 levels for 4 bits)
        # Optimized for standard normal distribution
        self.nf4_bins = torch.tensor([
            -1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453,
            -0.28444138169288635, -0.18477343022823334, -0.09105003625154495, 0.0,
            0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
            0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0
        ])
        
        self.code_to_bin = {i: self.nf4_bins[i] for i in range(16)}
    
    def quantize(self, weights):
        """Quantize weights to 4-bit NF4."""
        # Normalize to [-1, 1] range (store absmax)
        absmax = weights.abs().max()
        weights_normalized = weights / (absmax + 1e-8)
        
        # Quantize: find closest bin for each weight
        quantized = torch.zeros_like(weights, dtype=torch.int8)
        for i in range(16):
            if i < 15:
                mask = (weights_normalized >= self.nf4_bins[i]) & (weights_normalized < self.nf4_bins[i+1])
            else:
                mask = weights_normalized >= self.nf4_bins[i]
            quantized[mask] = i
        
        return quantized, absmax
    
    def dequantize(self, quantized, absmax):
        """Dequantize from 4-bit to full precision."""
        dequantized = torch.zeros_like(quantized, dtype=torch.float32)
        for code, bin_value in self.code_to_bin.items():
            mask = quantized == code
            dequantized[mask] = bin_value
        
        # Denormalize
        dequantized = dequantized * absmax
        
        return dequantized

# Demonstrate NF4 quantization
nf4 = NF4Quantization()

# Create sample weights (normally distributed)
weights = torch.randn(1000, 1000)
print(f"Original weights: {weights.element_size() * weights.numel() / 1024**2:.2f} MB")

# Quantize
quantized, absmax = nf4.quantize(weights)
print(f"Quantized (4-bit): {quantized.element_size() * quantized.numel() / 8 / 1024**2:.2f} MB")
print(f"Compression: {(weights.element_size() * 8) / 4:.0f}×")

# Dequantize
reconstructed = nf4.dequantize(quantized, absmax)

# Measure error
mse = F.mse_loss(weights, reconstructed)
print(f"\nReconstruction MSE: {mse.item():.6f}")
print(f"Relative error: {(mse.sqrt() / weights.std()).item()*100:.2f}%")

# Visualize quantization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Original distribution
axes[0].hist(weights.flatten().numpy(), bins=50, alpha=0.7, color='blue')
axes[0].set_title('Original Weights (FP32)')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.3)

# Quantized bins
axes[1].scatter(range(16), nf4.nf4_bins.numpy(), s=100, c='red', marker='x')
axes[1].set_title('NF4 Quantization Bins')
axes[1].set_xlabel('Bin Index (4-bit code)')
axes[1].set_ylabel('Bin Value')
axes[1].grid(alpha=0.3)

# Reconstruction error
error = (weights - reconstructed).flatten().numpy()
axes[2].hist(error, bins=50, alpha=0.7, color='orange')
axes[2].set_title('Quantization Error')
axes[2].set_xlabel('Error')
axes[2].set_ylabel('Frequency')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Prefix Tuning

**Idea**: Prepend trainable "prefix" tokens to input

Instead of modifying weights, optimize prefix embeddings:

$$[P_1, P_2, ..., P_k, x_1, x_2, ..., x_n]$$

where $P_i$ are trainable virtual tokens.

**Advantages**:
- Even fewer parameters than LoRA
- One frozen model + multiple prefixes

In [ ]:
class PrefixTuning(nn.Module):
    """
    Prefix tuning: prepend trainable prefix to keys and values.
    
    For each layer, learn prefix: P_K (prefix keys), P_V (prefix values)
    Attention over [P_K | K], [P_V | V]
    """
    
    def __init__(self, prefix_length, d_model, n_heads, n_layers):
        super().__init__()
        
        self.prefix_length = prefix_length
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        
        # Trainable prefix embeddings for each layer (keys and values)
        # Shape: (n_layers, 2, n_heads, prefix_length, d_model // n_heads)
        self.prefix_embeddings = nn.Parameter(
            torch.randn(n_layers, 2, n_heads, prefix_length, d_model // n_heads)
        )
        
    def get_prefix(self, layer_idx, batch_size):
        """Get prefix keys and values for specific layer."""
        # (2, n_heads, prefix_length, d_k)
        prefix = self.prefix_embeddings[layer_idx]
        
        # Expand for batch
        # (batch, n_heads, prefix_length, d_k)
        prefix_keys = prefix[0].unsqueeze(0).expand(batch_size, -1, -1, -1)
        prefix_values = prefix[1].unsqueeze(0).expand(batch_size, -1, -1, -1)
        
        return prefix_keys, prefix_values
    
    def forward(self, keys, values, layer_idx):
        """
        Prepend prefix to keys and values.
        
        Args:
            keys: (batch, n_heads, seq_len, d_k)
            values: (batch, n_heads, seq_len, d_k)
            layer_idx: which transformer layer
        """
        batch_size = keys.size(0)
        
        # Get prefix for this layer
        prefix_keys, prefix_values = self.get_prefix(layer_idx, batch_size)
        
        # Concatenate prefix with actual keys/values
        keys = torch.cat([prefix_keys, keys], dim=2)  # (batch, n_heads, prefix_len + seq_len, d_k)
        values = torch.cat([prefix_values, values], dim=2)
        
        return keys, values

# Create prefix tuning
d_model = 768
n_heads = 12
n_layers = 12
prefix_length = 10

prefix_model = PrefixTuning(prefix_length, d_model, n_heads, n_layers)

# Count parameters
total_model_params = 100e6  # Assume 100M parameter base model
prefix_params = sum(p.numel() for p in prefix_model.parameters())

print(f"Base model parameters: {total_model_params/1e6:.0f}M")
print(f"Prefix tuning parameters: {prefix_params:,}")
print(f"Trainable fraction: {prefix_params/total_model_params*100:.4f}%")

# Test
batch_size = 2
seq_len = 128
d_k = d_model // n_heads

keys = torch.randn(batch_size, n_heads, seq_len, d_k)
values = torch.randn(batch_size, n_heads, seq_len, d_k)

keys_with_prefix, values_with_prefix = prefix_model(keys, values, layer_idx=0)

print(f"\nOriginal keys shape: {keys.shape}")
print(f"With prefix: {keys_with_prefix.shape}")
print(f"Added prefix length: {keys_with_prefix.size(2) - keys.size(2)}")

## 7. Adapter Layers

**Idea**: Insert small trainable modules between frozen layers

Architecture:
```
x → LayerNorm → Attention → [Adapter] → Add
  ↓                                      ↑
  └──────────────────────────────────────┘
```

Adapter: Down-project → ReLU → Up-project

In [ ]:
class Adapter(nn.Module):
    """
    Adapter layer: bottleneck architecture.
    
    Reduce dimension → nonlinearity → expand dimension
    """
    
    def __init__(self, d_model, bottleneck_dim, dropout=0.1):
        super().__init__()
        
        self.down_proj = nn.Linear(d_model, bottleneck_dim)
        self.up_proj = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Initialize near identity (small random weights)
        nn.init.xavier_uniform_(self.down_proj.weight, gain=0.01)
        nn.init.zeros_(self.up_proj.weight)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.zeros_(self.up_proj.bias)
    
    def forward(self, x):
        # x: (batch, seq, d_model)
        residual = x
        
        # Bottleneck
        x = self.down_proj(x)  # (batch, seq, bottleneck_dim)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.up_proj(x)  # (batch, seq, d_model)
        
        # Residual connection
        return residual + x

class TransformerBlockWithAdapter(nn.Module):
    """Transformer block with adapter after attention."""
    
    def __init__(self, d_model, n_heads, d_ff, bottleneck_dim):
        super().__init__()
        
        self.attention = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.adapter = Adapter(d_model, bottleneck_dim)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        
        # Freeze attention and FFN, train only adapter
        for param in self.attention.parameters():
            param.requires_grad = False
        for param in self.ffn.parameters():
            param.requires_grad = False
    
    def forward(self, x):
        # Self-attention
        attn_out, _ = self.attention(x, x, x)
        x = self.norm1(x + attn_out)
        
        # Adapter (trainable)
        x = self.adapter(x)
        
        # FFN
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x

# Create block with adapter
d_model = 768
n_heads = 12
d_ff = 3072
bottleneck_dim = 64

block = TransformerBlockWithAdapter(d_model, n_heads, d_ff, bottleneck_dim)

# Count parameters
total_params = sum(p.numel() for p in block.parameters())
trainable_params = sum(p.numel() for p in block.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters (adapter only): {trainable_params:,}")
print(f"Trainable: {trainable_params/total_params*100:.2f}%")

# Test
x = torch.randn(2, 128, d_model)
output = block(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

## 8. Comparing PEFT Methods

In [ ]:
def compare_peft_methods():
    """Compare different PEFT approaches."""
    
    # Assume 1B parameter model
    base_params = 1000  # Million
    
    methods = {
        'Full Fine-tuning': {
            'params': base_params,
            'memory': base_params * 4 * 4,  # Model + grads + optimizer
            'performance': 100,
            'flexibility': 100
        },
        'LoRA (r=8)': {
            'params': base_params * 0.004,  # 0.4%
            'memory': base_params * 4 * 1.01,
            'performance': 95,
            'flexibility': 80
        },
        'QLoRA (r=8, 4-bit)': {
            'params': base_params * 0.004,
            'memory': base_params * 0.5 * 1.05,  # 4-bit base
            'performance': 93,
            'flexibility': 75
        },
        'Prefix Tuning': {
            'params': base_params * 0.001,  # ~0.1%
            'memory': base_params * 4,
            'performance': 85,
            'flexibility': 60
        },
        'Adapter': {
            'params': base_params * 0.01,  # ~1%
            'memory': base_params * 4 * 1.02,
            'performance': 90,
            'flexibility': 70
        }
    }
    
    # Extract data
    method_names = list(methods.keys())
    trainable_params = [methods[m]['params'] for m in method_names]
    memory = [methods[m]['memory'] for m in method_names]
    performance = [methods[m]['performance'] for m in method_names]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Trainable parameters
    axes[0, 0].barh(method_names, trainable_params, color='skyblue')
    axes[0, 0].set_xlabel('Trainable Parameters (M)')
    axes[0, 0].set_title('Trainable Parameters')
    axes[0, 0].set_xscale('log')
    axes[0, 0].grid(axis='x', alpha=0.3)
    
    # Memory
    axes[0, 1].barh(method_names, memory, color='lightcoral')
    axes[0, 1].set_xlabel('Peak Memory (GB)')
    axes[0, 1].set_title('Memory Requirements')
    axes[0, 1].set_xscale('log')
    axes[0, 1].grid(axis='x', alpha=0.3)
    
    # Performance
    axes[1, 0].barh(method_names, performance, color='lightgreen')
    axes[1, 0].set_xlabel('Relative Performance (%)')
    axes[1, 0].set_title('Task Performance')
    axes[1, 0].set_xlim(0, 105)
    axes[1, 0].grid(axis='x', alpha=0.3)
    
    # Efficiency frontier (performance vs parameters)
    axes[1, 1].scatter(trainable_params, performance, s=200, alpha=0.6)
    for i, name in enumerate(method_names):
        axes[1, 1].annotate(name.split()[0], (trainable_params[i], performance[i]),
                          xytext=(10, 0), textcoords='offset points', fontsize=9)
    axes[1, 1].set_xlabel('Trainable Parameters (M)')
    axes[1, 1].set_ylabel('Performance (%)')
    axes[1, 1].set_title('Efficiency Frontier')
    axes[1, 1].set_xscale('log')
    axes[1, 1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print("PEFT Method Recommendations:")
    print("="*70)
    print("\n1. LoRA (r=8-16)")
    print("   Best for: General fine-tuning, good balance")
    print("   Pros: Efficient, high performance, easy to merge")
    print("   Use when: You have GPU with >8GB VRAM")
    
    print("\n2. QLoRA (r=8, 4-bit)")
    print("   Best for: Large models on limited hardware")
    print("   Pros: Extremely memory efficient")
    print("   Use when: Limited GPU memory (<8GB)")
    
    print("\n3. Prefix Tuning")
    print("   Best for: Multiple task adapters")
    print("   Pros: Fewest parameters, fast switching")
    print("   Use when: Need many task-specific adaptations")
    
    print("\n4. Adapters")
    print("   Best for: Modular task-specific layers")
    print("   Pros: Interpretable, composable")
    print("   Use when: Want modular architecture")
    print("="*70)

compare_peft_methods()

## Summary

You've mastered parameter-efficient fine-tuning:
- ✅ Full fine-tuning baseline
-✅ LoRA: Low-rank adaptation (100-1000× parameter reduction)
- ✅ QLoRA: Quantized LoRA (4-bit base model)
- ✅ Prefix tuning: Trainable prefix tokens
- ✅ Adapters: Bottleneck layers

**Key Insights**:
1. **LoRA** decomposes weight updates: $\Delta W = BA$ where $r \ll d$
2. **Rank selection**: r=8 is sweet spot (0.4% params, ~95% performance)
3. **QLoRA** combines 4-bit base + FP16 adapters (4× memory reduction)
4. **Prefix tuning** has fewest parameters but lower performance ceiling
5. **Adapters** are modular and interpretable

**Best Practices**:
1. Start with LoRA (r=8, α=16) - works for most tasks
2. Use QLoRA if memory-constrained
3. Apply LoRA to Q and V projections (sometimes K too)
4. Tune rank based on task complexity
5. Consider multiple adapters for multiple tasks

**Next Module**: Practical fine-tuning with HuggingFace PEFT library

## Exercises

1. **Implement LoRA for FFN**: Add LoRA to feedforward layers
2. **Rank ablation**: Test r=[2,4,8,16,32] on classification task
3. **Adapter configuration**: Compare different bottleneck dimensions